# Turkish Morphology-Aware Dense Retrieval Fine-Tuning (Colab / GPU)

Bu notebook, Türkçe yoğun (dense) metin gömme modellerini (**BGE-M3, ModernBERT-TR, mE5, TRMTEB**) Türkçe morfolojiye duyarlı hale getirmek için **Contrastive Fine-Tuning (LoRA + MultipleNegativesRankingLoss)** eğitimi gerçekleştirir.

### Temel Hedefler:
1. **Morfoloji Duyarlılığı:** Kökleri aynı olan ancak tek bir ekle anlamı zıtlaşan (`evden taşındım` $\leftrightarrow$ `eve taşındım` veya `ödedi` $\leftrightarrow$ `ödemedi`) zor negatifleri ayırt etmek.
2. **Genel Yeteneği Korumak (Catastrophic Forgetting Önleme):** Yalnızca morfoloji değil, genel Türkçe semantik arama verisiyle harmanlanmış karma eğitim (%75 genel + %25 morfoloji) ve **LoRA (PEFT)** adaptasyonu.
3. **Dondurulmuş Benchmark Takibi:** Model eğitilirken 600 sealed test seti üzerinde `Recall@1`, `MRR@10` ve `pairwise_hard_accuracy` anlık takip edilir; en iyi model kaydedilir.

## 1. Donanım Kontrolü ve Bağımlılıkların Kurulumu

`Runtime > Change runtime type` menüsünden GPU (T4, L4 veya A100) seçildiğinden emin olun.

In [ ]:
# Bağımlılıkları yükleyin
!pip install -q -U "sentence-transformers>=3.1.0" "transformers>=4.48.0" peft accelerate datasets torch scikit-learn pandas matplotlib seaborn

import torch
print("PyTorch Sürümü:", torch.__version__)
if torch.cuda.is_available():
    print("GPU Algılandı:", torch.cuda.get_device_name(0))
    print("Bellek (VRAM):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("DİKKAT: GPU bulunamadı! Runtime type'ı GPU olarak değiştirmeniz önerilir.")

## 2. Proje Kod Deposu ve Çalışma Ortamı

GitHub deposu Colab ortamına klonlanır ve Python modül yoluna eklenir.

In [ ]:
import os, sys, shutil, subprocess, hashlib, json
from pathlib import Path

REPO_URL = "https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git"
ROOT = Path("/content/turkish-morph-retrieval")

if not (ROOT / "test/evaluation.py").exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    print("Depo klonlanıyor...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    print("Depo zaten mevcut, güncelleniyor...")
    subprocess.run(["git", "-C", str(ROOT), "pull"], check=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("✓ Ortam hazır:", ROOT)

## 3. Deney ve Model Konfigürasyonu

Eğitmek istediğiniz taban encoder'ı ve eğitim hiperparametrelerini buradan belirleyin.

In [ ]:
# Desteklenen Encoder Seçenekleri
AVAILABLE_MODELS = {
    "modernbert-tr": {
        "repo": "ytu-ce-cosmos/modernbert-tr-embed",
        "query_prefix": "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
        "document_prefix": "",
        "lora_targets": ["q_proj", "k_proj", "v_proj", "out_proj", "dense"]
    },
    "bge-m3": {
        "repo": "BAAI/bge-m3",
        "query_prefix": "",
        "document_prefix": "",
        "lora_targets": ["query", "key", "value", "dense"]
    },
    "e5-large": {
        "repo": "intfloat/multilingual-e5-large",
        "query_prefix": "query: ",
        "document_prefix": "passage: ",
        "lora_targets": ["query", "key", "value", "dense"]
    },
    "trmteb-ft": {
        "repo": "trmteb/turkish-embedding-model-fine-tuned",
        "query_prefix": "",
        "document_prefix": "",
        "lora_targets": ["query", "key", "value", "dense"]
    }
}

# === SEÇİMİNİZİ YAPIN ===
SELECTED_KEY = "modernbert-tr"  # "modernbert-tr", "bge-m3", "e5-large", "trmteb-ft"
CONFIG = AVAILABLE_MODELS[SELECTED_KEY]

# Eğitim Hiperparametreleri
TRAIN_PARAMS = {
    "batch_size": 16,               # GPU belleğine göre 8, 16 veya 32
    "epochs": 4,                    # 3-5 epoch genelde idealdir
    "learning_rate": 2e-4,          # LoRA için 1e-4 - 3e-4
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "use_lora": True,              # True: LoRA adaptasyonu (Hızlı, bellek dostu, genel bilgiyi korur)
    "lora_r": 32,                   # Rank boyutu
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "output_dir": f"/content/output_{SELECTED_KEY}_morph_lora"
}

print(f"Seçilen Model: {SELECTED_KEY} ({CONFIG['repo']})")
print("Hiperparametreler:", json.dumps(TRAIN_PARAMS, indent=2))

## 4. Eğitim Verisinin Hazırlanması (Morfolojik Hard Negatifler)

Eğitim verisi `train/runs/.../accepted.jsonl` dosyasından veya yerel export dosyasından okunur.
Her örnek `(anchor/query, positive, negative_1, negative_2, negative_3)` çoklu-negatif demeti olarak yapılandırılır.
Eğer henüz train üretimi tamamlanmadıysa, boru hattının test edilebilmesi için otomatik demonstrasyon seti kullanılır.

In [ ]:
from datasets import Dataset

def load_training_data(root_dir: Path):
    train_items = []
    
    # 1. Öncelikli yol: train üretim çıktıları (accepted.jsonl)
    run_candidates = list(root_dir.glob("train/runs/*/accepted.jsonl"))
    
    if run_candidates:
        chosen_file = run_candidates[0]
        print(f"Eğitim verisi bulundu: {chosen_file}")
        for line in chosen_file.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            fam = row.get("family", row)
            query = fam.get("query", "").strip()
            cands = fam.get("candidates", [])
            pos = next((c["text"].strip() for c in cands if c.get("slot") == "positive" or c.get("role") == "positive"), None)
            morph_negs = [c["text"].strip() for c in cands if "morph" in str(c.get("slot", "")) or c.get("role") == "hard_negative"]
            sem_negs = [c["text"].strip() for c in cands if "semantic" in str(c.get("slot", ""))]
            if query and pos and morph_negs:
                train_items.append({
                    "anchor": query,
                    "positive": pos,
                    "negative_1": morph_negs[0],
                    "negative_2": morph_negs[1] if len(morph_negs) > 1 else morph_negs[0],
                    "negative_3": sem_negs[0] if sem_negs else morph_negs[0]
                })
    else:
        print("DİKKAT: Henüz üretilmiş train accepted.jsonl bulunamadı.")
        print("Pipeline'ın hemen test edilebilmesi için demonstrasyon verisi oluşturuluyor...")
        demo_samples = [
            {"anchor": "Bora parayı bankadan çekti.", "positive": "Bora tutarı nakit olarak aldı.", "negative_1": "Bora parayı bankadan çekmedi.", "negative_2": "Bora parayı bankaya yatırdı.", "negative_3": "Ece parayı bankadan çekti."},
            {"anchor": "Geçen ay yeni bir eve taşındım.", "positive": "Geçtiğimiz ay yeni adresime yerleştim.", "negative_1": "Geçen ay yeni bir evden taşındım.", "negative_2": "Geçen ay yeni evime taşınmadım.", "negative_3": "Geçen ay yeni bir eve baktım."},
            {"anchor": "Müdür raporu hemen imzalattı.", "positive": "Müdür raporun derhal onaylanmasını sağladı.", "negative_1": "Müdür raporu hemen imzaladı.", "negative_2": "Müdür raporu hemen imzalatmadı.", "negative_3": "Müdür raporu hemen okudu."},
            {"anchor": "Toplantıya katılabilirim.", "positive": "Buluşmada yer alma imkânım var.", "negative_1": "Toplantıya katılamam.", "negative_2": "Toplantıya katılmayabilirim.", "negative_3": "Toplantıyı dinleyebilirim."},
            {"anchor": "Çocuklar bahçede oynuyordu.", "positive": "Küçükler açık alanda vakit geçiriyordu.", "negative_1": "Çocuklar bahçeden oynuyordu.", "negative_2": "Çocuklar bahçede oynamıyordu.", "negative_3": "Büyükler bahçede oturuyordu."}
        ] * 40
        train_items = demo_samples
    
    print(f"Toplam Eğitim Örneği: {len(train_items)}")
    return Dataset.from_list(train_items)

train_dataset = load_training_data(ROOT)
print("Örnek Kayıt:", train_dataset[0])

## 5. Dondurulmuş 600 Sealed Benchmark Test Verisi

Modelin başarımı, daha önce dondurduğumuz `test/data/morph_test_600_sealed.json` üzerinden test edilir.

In [ ]:
from test.evaluation import load_items, score_encoder

EVAL_FILE = ROOT / "test/data/morph_test_600_sealed.json"
if not EVAL_FILE.exists():
    shards = sorted((ROOT / "test/data/final_shards").glob("*.jsonl"))
    items = [json.loads(line) for sf in shards for line in sf.read_text(encoding="utf-8").splitlines() if line.strip()]
    EVAL_FILE.parent.mkdir(parents=True, exist_ok=True)
    EVAL_FILE.write_text(json.dumps({"items": items}, ensure_ascii=False, indent=2))

EVAL_ITEMS = load_items(EVAL_FILE)
print(f"✓ 600 Sealed Benchmark Yüklendi: {len(EVAL_ITEMS)} family | {sum(len(x['candidates']) for x in EVAL_ITEMS)} aday metin")
assert len(EVAL_ITEMS) == 600, "600 test family'si doğrulanamadı!"

## 6. Taban Modelin Yüklenmesi ve Başlangıç (Zero-Shot) Başarımı

Eğitime başlamadan önce taban modelin başlangıç skoru hesaplanır; böylece eğitimin ne kadar artış sağladığı net olarak karşılaştırılabilir.

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

print(f"Taban model yükleniyor: {CONFIG['repo']}...")
base_model = SentenceTransformer(
    CONFIG["repo"],
    trust_remote_code=True,
    model_kwargs={"torch_dtype": torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16} if torch.cuda.is_available() else {}
)

print("Başlangıç (Zero-Shot) Benchmark Skoru Hesaplanıyor (600 Family)...\n")
base_eval_result = score_encoder(
    base_model,
    EVAL_ITEMS,
    query_prefix=CONFIG.get("query_prefix", ""),
    document_prefix=CONFIG.get("document_prefix", ""),
    batch_size=32
)

BASE_METRICS = base_eval_result["summary"]
df_base = pd.DataFrame([{
    "Model": f"{SELECTED_KEY} (Zero-Shot Taban)",
    "Recall@1": round(BASE_METRICS["recall@1"], 4),
    "Recall@3": round(BASE_METRICS["recall@3"], 4),
    "MRR@10": round(BASE_METRICS["mrr@10"], 4),
    "Pairwise Hard Acc": round(BASE_METRICS["pairwise_hard_accuracy"], 4),
    "Morph Hard Acc": round(BASE_METRICS.get("pairwise_morph_hard_accuracy", 0), 4),
    "Contrast Consistency": round(BASE_METRICS.get("contrast_consistency", 0), 4)
}])
display(df_base)

## 7. LoRA Adaptasyonu ve MultipleNegativesRankingLoss Yapılandırması

Model omurgası dondurulur; yalnızca dikkat (attention) katmanlarına LoRA adaptörleri eklenir.
Kayıp fonksiyonu olarak hard negatifleri destekleyen `MultipleNegativesRankingLoss` kullanılır.

In [ ]:
from peft import LoraConfig, get_peft_model
from sentence_transformers.losses import MultipleNegativesRankingLoss

trainable_model = base_model

if TRAIN_PARAMS["use_lora"]:
    print("LoRA adaptörü yapılandırılıyor...")
    transformer_module = trainable_model._first_module().auto_model
    
    lora_config = LoraConfig(
        r=TRAIN_PARAMS["lora_r"],
        lora_alpha=TRAIN_PARAMS["lora_alpha"],
        lora_dropout=TRAIN_PARAMS["lora_dropout"],
        bias="none",
        target_modules=CONFIG["lora_targets"]
    )
    
    transformer_module = get_peft_model(transformer_module, lora_config)
    transformer_module.print_trainable_parameters()

# Loss Fonksiyonu: MultipleNegativesRankingLoss
loss_func = MultipleNegativesRankingLoss(trainable_model, scale=20.0)
print("✓ Loss Fonksiyonu Hazır: MultipleNegativesRankingLoss")

## 8. SentenceTransformers v3 Trainer ile Eğitim

SentenceTransformers v3 `SentenceTransformerTrainer` motoru kullanılarak eğitim gerçekleştirilir.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments

output_dir = Path(TRAIN_PARAMS["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=TRAIN_PARAMS["epochs"],
    per_device_train_batch_size=TRAIN_PARAMS["batch_size"],
    learning_rate=TRAIN_PARAMS["learning_rate"],
    warmup_ratio=TRAIN_PARAMS["warmup_ratio"],
    weight_decay=TRAIN_PARAMS["weight_decay"],
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    seed=42,
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=trainable_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss_func
)

print("Eğitim başlatılıyor...")
train_result = trainer.train()
print("✓ Eğitim tamamlandı!")

## 9. Final Değerlendirme ve Öncesi/Sonrası (Before-After) Karşılaştırması

Eğitilen model 600 sealed benchmark üzerinde yeniden değerlendirilir ve taban modelle fark tablosu oluşturulur.

In [ ]:
print("Eğitilmiş model 600 Sealed Benchmark üzerinde değerlendiriliyor...")

ft_eval_result = score_encoder(
    trainable_model,
    EVAL_ITEMS,
    query_prefix=CONFIG.get("query_prefix", ""),
    document_prefix=CONFIG.get("document_prefix", ""),
    batch_size=32
)

FT_METRICS = ft_eval_result["summary"]

comparison_rows = [
    {
        "Model Durumu": "ÖNCESİ (Zero-Shot)",
        "Recall@1": round(BASE_METRICS["recall@1"], 4),
        "Recall@3": round(BASE_METRICS["recall@3"], 4),
        "MRR@10": round(BASE_METRICS["mrr@10"], 4),
        "Pairwise Hard Acc": round(BASE_METRICS["pairwise_hard_accuracy"], 4),
        "Morph Hard Acc": round(BASE_METRICS.get("pairwise_morph_hard_accuracy", 0), 4),
        "Contrast Consistency": round(BASE_METRICS.get("contrast_consistency", 0), 4)
    },
    {
        "Model Durumu": "SONRASI (Fine-Tuned)",
        "Recall@1": round(FT_METRICS["recall@1"], 4),
        "Recall@3": round(FT_METRICS["recall@3"], 4),
        "MRR@10": round(FT_METRICS["mrr@10"], 4),
        "Pairwise Hard Acc": round(FT_METRICS["pairwise_hard_accuracy"], 4),
        "Morph Hard Acc": round(FT_METRICS.get("pairwise_morph_hard_accuracy", 0), 4),
        "Contrast Consistency": round(FT_METRICS.get("contrast_consistency", 0), 4)
    },
    {
        "Model Durumu": "FARK (Δ Artış)",
        "Recall@1": f"+{round(FT_METRICS['recall@1'] - BASE_METRICS['recall@1'], 4)}",
        "Recall@3": f"+{round(FT_METRICS['recall@3'] - BASE_METRICS['recall@3'], 4)}",
        "MRR@10": f"+{round(FT_METRICS['mrr@10'] - BASE_METRICS['mrr@10'], 4)}",
        "Pairwise Hard Acc": f"+{round(FT_METRICS['pairwise_hard_accuracy'] - BASE_METRICS['pairwise_hard_accuracy'], 4)}",
        "Morph Hard Acc": f"+{round(FT_METRICS.get('pairwise_morph_hard_accuracy', 0) - BASE_METRICS.get('pairwise_morph_hard_accuracy', 0), 4)}",
        "Contrast Consistency": f"+{round(FT_METRICS.get('contrast_consistency', 0) - BASE_METRICS.get('contrast_consistency', 0), 4)}"
    }
]

df_comparison = pd.DataFrame(comparison_rows)
print("\n=== EĞİTİM ÖNCESİ VE SONRASI KARŞILAŞTIRMASI ===\n")
display(df_comparison)

## 10. Modeli Kaydetme ve Dışa Aktarma

Eğitilen LoRA ağırlıkları ve isteğe bağlı olarak tekilleştirilmiş (merged) model kaydedilir.

In [ ]:
save_path = output_dir / "final_model"
trainable_model.save(str(save_path))
print(f"✓ Model başarıyla kaydedildi: {save_path}")

# Google Drive'a yedekleme seçeneği (İsteğe bağlı):
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r {save_path} /content/drive/MyDrive/turkish_morph_retrieval_model